# Day 6 — Advanced Analytics

Bluestock Mutual Fund Analytics Capstone — VaR/CVaR, Rolling Sharpe, Investor Cohorts, SIP Continuity, Fund Recommender and Sector Concentration.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

BASE_DIR = Path('..')
RAW = BASE_DIR / 'data' / 'raw'
OUT = BASE_DIR / 'outputs'
CHARTS = BASE_DIR / 'charts' / 'day6'
OUT.mkdir(exist_ok=True)
CHARTS.mkdir(parents=True, exist_ok=True)

nav = pd.read_csv(RAW/'02_nav_history.csv')
fund = pd.read_csv(RAW/'01_fund_master.csv')
perf = pd.read_csv(RAW/'07_scheme_performance.csv')
transactions = pd.read_csv(RAW/'08_investor_transactions.csv')
holdings = pd.read_csv(RAW/'09_portfolio_holdings.csv')

print(nav.shape, fund.shape, perf.shape, transactions.shape, holdings.shape)

## 1. Daily Returns and Historical VaR/CVaR

Historical VaR at 95% confidence is calculated as the 5th percentile of daily returns. CVaR is the average return below the VaR threshold.

In [ ]:
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
nav['nav'] = pd.to_numeric(nav['nav'], errors='coerce')
nav = nav.dropna(subset=['date','amfi_code','nav']).sort_values(['amfi_code','date']).drop_duplicates(['amfi_code','date'])
nav = nav[nav['nav'] > 0]
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
returns = nav.dropna(subset=['daily_return']).merge(fund[['amfi_code','scheme_name','fund_house','category','sub_category','risk_category']], on='amfi_code', how='left')

var_rows = []
for code, g in returns.groupby('amfi_code'):
    r = g['daily_return'].dropna()
    var95 = r.quantile(0.05)
    cvar95 = r[r <= var95].mean()
    row = g.iloc[0]
    var_rows.append({
        'amfi_code': code,
        'scheme_name': row['scheme_name'],
        'fund_house': row['fund_house'],
        'category': row['category'],
        'risk_category': row['risk_category'],
        'observations': len(r),
        'historical_var_95': var95,
        'historical_cvar_95': cvar95,
        'var_95_pct': var95 * 100,
        'cvar_95_pct': cvar95 * 100
    })

var_cvar = pd.DataFrame(var_rows).sort_values('historical_var_95')
var_cvar.to_csv(OUT/'var_cvar_report.csv', index=False)
var_cvar.head()

**Insight 1:** `SBI Small Cap Fund - Direct Plan - Growth` shows the highest downside risk based on Historical VaR 95%. Supporting chart: `var_cvar_worst_funds.png`.

In [ ]:
plt.figure(figsize=(12,6))
vc = var_cvar.head(15)
labels = vc['scheme_name'].str.replace(' - Regular Plan - Growth','', regex=False).str.replace(' - Direct Plan - Growth','', regex=False)
plt.barh(labels, vc['var_95_pct'])
plt.title('Worst 15 Funds by Historical VaR 95%')
plt.xlabel('VaR 95% Daily Return (%)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHARTS/'var_cvar_worst_funds.png', dpi=300, bbox_inches='tight')
plt.show()

## 2. Rolling 90-Day Sharpe Ratio

Rolling Sharpe shows how risk-adjusted return changes over time. Risk-free rate used: 6.5% annual.

In [ ]:
risk_free_annual = 0.065
risk_free_daily = risk_free_annual / 252
key_codes = [119551, 120503, 118632, 119092, 120841]

plt.figure(figsize=(14,7))
rolling_output = []
for code in key_codes:
    g = nav[nav['amfi_code'] == code][['date','daily_return']].dropna().copy()
    g['rolling_90d_sharpe'] = ((g['daily_return'].rolling(90).mean() - risk_free_daily) / g['daily_return'].rolling(90).std()) * np.sqrt(252)
    scheme_name = fund.loc[fund['amfi_code'].eq(code), 'scheme_name'].iloc[0]
    plt.plot(g['date'], g['rolling_90d_sharpe'], label=scheme_name[:30])
    g['amfi_code'] = code
    g['scheme_name'] = scheme_name
    rolling_output.append(g)

rolling_sharpe = pd.concat(rolling_output)
rolling_sharpe.to_csv(OUT/'rolling_90d_sharpe.csv', index=False)
plt.axhline(0, linewidth=1)
plt.title('Rolling 90-Day Sharpe Ratio - 5 Key Funds')
plt.xlabel('Date')
plt.ylabel('Rolling Sharpe')
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(CHARTS/'rolling_sharpe_chart.png', dpi=300, bbox_inches='tight')
plt.show()

**Insight 2:** Rolling Sharpe helps identify time periods where fund returns were not enough to compensate for volatility. Supporting chart: `rolling_sharpe_chart.png`.

## 3. Investor Cohort Analysis

Cohorts are grouped by each investor's first transaction year.

In [ ]:
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'], errors='coerce')
transactions['amount_inr'] = pd.to_numeric(transactions['amount_inr'], errors='coerce')
transactions = transactions.dropna(subset=['transaction_date','investor_id','amount_inr'])
transactions['transaction_type_std'] = transactions['transaction_type'].astype(str).str.strip().str.lower().map({
    'sip':'SIP','lumpsum':'Lumpsum','lump sum':'Lumpsum','redemption':'Redemption','redeem':'Redemption'
}).fillna(transactions['transaction_type'].astype(str).str.title())

first_year = transactions.groupby('investor_id')['transaction_date'].min().dt.year.rename('first_transaction_year')
tx = transactions.merge(first_year, on='investor_id', how='left').merge(fund[['amfi_code','scheme_name']], on='amfi_code', how='left')
invested = tx[tx['transaction_type_std'].isin(['SIP','Lumpsum'])]
sip_only = tx[tx['transaction_type_std'].eq('SIP')]

cohort_base = invested.groupby('first_transaction_year').agg(
    investors=('investor_id','nunique'),
    total_invested=('amount_inr','sum'),
    avg_transaction_amount=('amount_inr','mean')
).reset_index()
avg_sip = sip_only.groupby('first_transaction_year')['amount_inr'].mean().rename('avg_sip_amount').reset_index()
top_pref = invested.groupby(['first_transaction_year','scheme_name'])['amount_inr'].sum().reset_index()
top_pref = top_pref.sort_values(['first_transaction_year','amount_inr'], ascending=[True,False]).drop_duplicates('first_transaction_year')
top_pref = top_pref[['first_transaction_year','scheme_name']].rename(columns={'scheme_name':'top_fund_preference'})
cohort = cohort_base.merge(avg_sip, on='first_transaction_year', how='left').merge(top_pref, on='first_transaction_year', how='left')
cohort.to_csv(OUT/'investor_cohort_analysis.csv', index=False)
cohort

**Insight 3:** The `2024` cohort contributed the highest invested amount. Supporting chart: `investor_cohort_total_invested.png`.

In [ ]:
plt.figure(figsize=(10,6))
plt.bar(cohort['first_transaction_year'].astype(str), cohort['total_invested']/1e7)
plt.title('Investor Cohort Total Invested')
plt.xlabel('First Transaction Year')
plt.ylabel('Total Invested (₹ Crore)')
plt.tight_layout()
plt.savefig(CHARTS/'investor_cohort_total_invested.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. SIP Continuity Analysis

Investors with 6+ SIP transactions are analysed. Investors are flagged as at-risk when the average or maximum gap exceeds 35 days.

In [ ]:
sip = sip_only.sort_values(['investor_id','transaction_date']).copy()
sip['gap_days'] = sip.groupby('investor_id')['transaction_date'].diff().dt.days
sip_counts = sip.groupby('investor_id').agg(
    sip_transaction_count=('transaction_date','count'),
    first_sip_date=('transaction_date','min'),
    last_sip_date=('transaction_date','max'),
    avg_gap_days=('gap_days','mean'),
    max_gap_days=('gap_days','max'),
    total_sip_amount=('amount_inr','sum')
).reset_index()
sip_continuity = sip_counts[sip_counts['sip_transaction_count'] >= 6].copy()
sip_continuity['at_risk'] = np.where((sip_continuity['avg_gap_days'] > 35) | (sip_continuity['max_gap_days'] > 35), 'Yes', 'No')
sip_continuity['continuity_status'] = np.where(sip_continuity['at_risk'].eq('Yes'), 'At Risk', 'Regular')
sip_continuity.to_csv(OUT/'sip_continuity_analysis.csv', index=False)
sip_continuity.head()

**Insight 4:** SIP continuity rate is `0.1%` among investors with 6+ SIP transactions. Supporting chart: `sip_continuity_pie.png`.

In [ ]:
status_counts = sip_continuity['continuity_status'].value_counts()
plt.figure(figsize=(8,6))
plt.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%')
plt.title('SIP Continuity: Regular vs At-Risk Investors')
plt.tight_layout()
plt.savefig(CHARTS/'sip_continuity_pie.png', dpi=300, bbox_inches='tight')
plt.show()

## 5. Simple Fund Recommender

Input risk appetite: Low / Moderate / High. Output top 3 funds by Sharpe Ratio within matching risk grade.

In [ ]:
RISK_MAP = {
    'Low': ['Low'],
    'Moderate': ['Moderate', 'Moderately High'],
    'High': ['High', 'Very High']
}

def recommend_funds(risk_appetite='Moderate', top_n=3):
    allowed = RISK_MAP.get(risk_appetite, [risk_appetite])
    result = perf[perf['risk_grade'].isin(allowed)].copy()
    result['sharpe_ratio'] = pd.to_numeric(result['sharpe_ratio'], errors='coerce')
    cols = ['amfi_code','scheme_name','fund_house','category','risk_grade','return_3yr_pct','sharpe_ratio','expense_ratio_pct','aum_crore']
    return result.sort_values('sharpe_ratio', ascending=False)[cols].head(top_n)

recommend_funds('Moderate')

## 6. Sector HHI Concentration

HHI = sum of squared portfolio weights. A higher HHI means higher concentration risk.

In [ ]:
holdings['weight_decimal'] = pd.to_numeric(holdings['weight_pct'], errors='coerce') / 100
hhi_report = holdings.dropna(subset=['weight_decimal']).groupby('amfi_code').agg(
    sector_hhi=('weight_decimal', lambda x: float(np.sum(np.square(x)))),
    holdings_count=('stock_symbol','nunique'),
    top_sector_weight_pct=('weight_pct','max')
).reset_index().merge(fund[['amfi_code','scheme_name','fund_house','category','sub_category','risk_category']], on='amfi_code', how='left')
hhi_report = hhi_report.sort_values('sector_hhi', ascending=False)
hhi_report.to_csv(OUT/'sector_hhi_concentration.csv', index=False)
hhi_report.head()

**Insight 5:** `Axis Bluechip Fund - Regular - Growth` has the highest HHI and therefore the highest concentration risk among funds with portfolio holdings. Supporting chart: `sector_hhi_top15.png`.

In [ ]:
plt.figure(figsize=(12,6))
hh = hhi_report.head(15)
labels = hh['scheme_name'].str.replace(' - Regular Plan - Growth','', regex=False).str.replace(' - Direct Plan - Growth','', regex=False)
plt.barh(labels, hh['sector_hhi'])
plt.title('Top 15 Funds by Sector HHI Concentration')
plt.xlabel('HHI')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(CHARTS/'sector_hhi_top15.png', dpi=300, bbox_inches='tight')
plt.show()

## Final Outputs

- `outputs/var_cvar_report.csv`
- `outputs/rolling_90d_sharpe.csv`
- `outputs/investor_cohort_analysis.csv`
- `outputs/sip_continuity_analysis.csv`
- `outputs/sector_hhi_concentration.csv`
- `scripts/recommender.py`
- `charts/day6/rolling_sharpe_chart.png`